# NeXo v3.0 · Notebook 01 — Preprocessing + Feature Engineering + Data Warehouse (CRISP-DM Phase 3)

> **CRISP-DM Phase 3 — Data Preparation.** Transforms raw TT_data/ CSVs into curated `warehouse.parquet` consumed by training notebooks 02 / 03 / 04.

## Contract

- **Input**: raw BSS + OSS CSVs from `TT_data/` (uploaded to MinIO `raw/` bucket).
- **Output**: `curated/warehouse.parquet` (subscriber + area-level features + CEM score target), `curated/cells.parquet` (per-cell for VAE), `curated/splits.json`.
- **Consumed by**: notebooks 02 (CEM), 03 (VAE), 04 (RAT) via NPZ exports.
- **Idempotent**: re-running with same SEED + same source data produces byte-identical parquets.

## Pipeline

```
TT_data/ CSVs
  → upload to MinIO raw/
  → read back from raw/
  → normalize OSS (parse %, derive area, 3GPP-formula latency/loss/jitter)
  → clean BSS (domain rules + clipping)
  → IterativeImputer (MICE-style) on numeric NaNs
  → Winsorize p99 on traffic columns (bounds saved)
  → derive features (data_intensity, rat-share, attach_gap, is_4g_capable, usim_bottleneck)
  → join BSS × OSS aggregates by area
  → compute CEM score target (0.4·attach + 0.3·4g_share + 0.2·integrity + 0.1·cdr_inv)
  → train/val/test split (random stratified by area + temporal holdout)
  → write curated/ to MinIO
```

## What this notebook does NOT do

- Train models — that's notebooks 02 / 03 / 04.
- Tune hyperparameters of the imputer / Winsorize threshold — that's a follow-up sensitivity-analysis pass.
- Validate against the Granger gate — that's notebook 10 (Tier 1 offline gate).


## 0 · Reproducibility, Constants & CEM Weights

**What this shows:** All hyperparameters in ONE place — SEED, CEM-score formula weights, Winsorize percentile, split fractions, imputer iteration cap. Papermill-overridable.

**What to look for:** The four `CEM_W_*` constants define the CEM score formula. These are the most important hyperparameters in the entire project. They MUST sum to 1.0 (asserted). Any change requires owner approval + sensitivity analysis.

In [1]:
# --- Papermill parameters cell (tag: parameters) ---
# Default constants. Override at retrain time via papermill -p flags.
# All magic numbers concentrated here for review.

SEED                       = 42        # numpy / random / sklearn / torch base seed

# --- BSS cleaning constants ---
ATTACH_SR_MIN              = 0.0       # success rate min bound
ATTACH_SR_MAX              = 1.0       # success rate max bound (clip to [0, 1])
WINSORIZE_PCT              = 99        # cap heavy-tailed traffic at p99 (justification: §8 ablation)

# --- CEM score weighting (cell 13) ---
# CEM score = 0.40 * attach_success + 0.30 * traffic_4g_share + 0.20 * data_integrity + 0.10 * cdr_inverse
# Sources for weights: domain expert (Huawei SmartCare scoring guidance, OSS+BSS convergence doc)
# These weights are the SINGLE most important hyperparameter in the project.
# Owner must approve any change. Sensitivity analysis recommended (see explainer).
CEM_W_ATTACH               = 0.40      # subscriber network-attach success rate
CEM_W_4G_SHARE             = 0.30      # share of traffic on 4G (modern RAT preference)
CEM_W_INTEGRITY            = 0.20      # OSS data-integrity area average
CEM_W_CDR_INV              = 0.10      # 1 - call-drop-rate area average

# --- Split policy ---
SPLIT_TRAIN_FRAC           = 0.70      # train share
SPLIT_VAL_FRAC             = 0.15      # validation share
SPLIT_TEST_FRAC            = 0.15      # test share (random) + last-10%-months temporal holdout
SPLIT_STRATIFY_COL         = 'area'    # ensures geographic representation
TEMPORAL_HOLDOUT_MONTHS_N  = 1         # how many trailing months reserved for temporal test

# --- Imputer ---
IMPUTER_MAX_ITER           = 10        # IterativeImputer convergence cap

# Hardcoded seed setup
import os, random
import numpy as np
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Assert weight sum to 1.0 (defense-grade contract)
assert abs(CEM_W_ATTACH + CEM_W_4G_SHARE + CEM_W_INTEGRITY + CEM_W_CDR_INV - 1.0) < 1e-9, \
    'CEM weights must sum to 1.0 (currently: {})'.format(CEM_W_ATTACH + CEM_W_4G_SHARE + CEM_W_INTEGRITY + CEM_W_CDR_INV)

print(f'SEED                       = {SEED}')
print(f'CEM weights (attach/4g/integrity/cdr_inv) = ({CEM_W_ATTACH}, {CEM_W_4G_SHARE}, {CEM_W_INTEGRITY}, {CEM_W_CDR_INV})')
print(f'CEM weight sum             = {CEM_W_ATTACH+CEM_W_4G_SHARE+CEM_W_INTEGRITY+CEM_W_CDR_INV}')
print(f'Winsorize percentile       = {WINSORIZE_PCT}')
print(f'Split (train/val/test)     = ({SPLIT_TRAIN_FRAC}, {SPLIT_VAL_FRAC}, {SPLIT_TEST_FRAC})'); print(f'Temporal holdout months    = {TEMPORAL_HOLDOUT_MONTHS_N}')

SEED                       = 42
CEM weights (attach/4g/integrity/cdr_inv) = (0.4, 0.3, 0.2, 0.1)
CEM weight sum             = 0.9999999999999999
Winsorize percentile       = 99
Split (train/val/test)     = (0.7, 0.15, 0.15)
Temporal holdout months    = 1


## 1 · Imports + MinIO client

In [2]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
from IPython.display import display

import io, json, os, warnings
from pathlib import Path
import boto3, joblib, numpy as np, pandas as pd
from botocore.client import Config
from botocore.exceptions import ClientError
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)

S3_ENDPOINT = os.environ.get('S3_ENDPOINT', 'http://localhost:9000')
s3 = boto3.client('s3', endpoint_url=S3_ENDPOINT,
                  aws_access_key_id=os.environ.get('S3_ACCESS_KEY','minio'),
                  aws_secret_access_key=os.environ.get('S3_SECRET_KEY','minio_pw'),
                  config=Config(signature_version='s3v4'))
RAW='raw'; PROC='processed'; CUR='curated'
print(f'MinIO endpoint = {S3_ENDPOINT}')

MinIO endpoint = http://localhost:9000


## 2 · Bootstrap MinIO buckets + PURGE raw of stale objects

Creates raw/processed/curated if missing. Then enforces invariant: `raw/` contains **only**
`bss/*` + `oss/*` keys. Anything else (old `run-*.json`, leftover synthetic) is deleted.

In [3]:
from time import sleep

def ensure(b):
    try: s3.head_bucket(Bucket=b); print(f'  exists: {b}')
    except ClientError: s3.create_bucket(Bucket=b); print(f'  CREATED: {b}')
for b in [RAW, PROC, CUR]:
    for attempt in range(5):
        try:
            ensure(b)
            break
        except Exception as e:
            if attempt == 4:
                raise
            print(f'  retrying {b} after S3 error: {type(e).__name__}')
            sleep(2)

# Purge raw of non-dataset keys
paginator = s3.get_paginator('list_objects_v2')
to_delete = []
kept = 0
for page in paginator.paginate(Bucket=RAW):
    for obj in page.get('Contents', []):
        k = obj['Key']
        if k.startswith(('bss/', 'oss/')): kept += 1
        else: to_delete.append({'Key': k})
if to_delete:
    for i in range(0, len(to_delete), 1000):
        s3.delete_objects(Bucket=RAW, Delete={'Objects': to_delete[i:i+1000], 'Quiet': True})
    print(f'  purged {len(to_delete)} stale objects ({kept} datasets retained)')
else:
    print(f'  raw/ already clean ({kept} dataset objects)')

  exists: raw
  exists: processed
  exists: curated
  raw/ already clean (3140 dataset objects)


## 3 · Upload BSS (16 files) + generated OSS (48 files) → raw/

- BSS: `TT_data/BSS/smartcare_cem_*.csv` → `raw/bss/`
- OSS: `TT_data/OSS/generated/oss_<rat>_<token>.csv` → `raw/oss/`

Idempotent — files with matching size are skipped.

In [4]:
ROOT = Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
TT_DATA = ROOT/'TT_data'

def upload(p, key):
    try:
        if s3.head_object(Bucket=RAW, Key=key)['ContentLength']==p.stat().st_size: return False
    except ClientError: pass
    s3.upload_file(str(p), RAW, key); return True

up=sk=0
for f in sorted((TT_DATA/'BSS').glob('smartcare_cem_*.csv')):
    if upload(f, f'bss/{f.name}'): up+=1; print(f'  ↑ raw/bss/{f.name}')
    else: sk+=1
for f in sorted((TT_DATA/'OSS'/'generated').glob('oss_*.csv')):
    if upload(f, f'oss/{f.name}'): up+=1; print(f'  ↑ raw/oss/{f.name}')
    else: sk+=1
print(f'\n{up} uploaded, {sk} skipped (already present).')


0 uploaded, 64 skipped (already present).


## 4 · Read raw BSS back from MinIO

All 16 BSS monthly CSVs stitched. Tags: `month_year` (e.g. `2026-03`), `source_origin`.

In [5]:
import re, gc

MONTH_MAP = {'jan':'2026-01','feb':'2026-02','mars':'2026-03','avr':'2026-04','mai':'2026-05',
             'jun':'2026-06','jul':'2026-07','aug':'2026-08','aou':'2026-08','sep':'2026-09'}
REAL = {'feb','mars'}

def list_keys(bucket, prefix):
    keys=[]
    for page in s3.get_paginator('list_objects_v2').paginate(Bucket=bucket, Prefix=prefix):
        for o in page.get('Contents', []): keys.append(o['Key'])
    return keys

def shrink(df, cat_thresh=0.5):
    """Downcast numerics + low-cardinality objects to category. ~50% RAM cut on big tables."""
    for c in df.select_dtypes(include=['float64']).columns:
        df[c] = pd.to_numeric(df[c], downcast='float')
    for c in df.select_dtypes(include=['int64']).columns:
        df[c] = pd.to_numeric(df[c], downcast='integer')
    for c in df.select_dtypes(include=['object']).columns:
        if df[c].nunique(dropna=False) / max(len(df), 1) < cat_thresh:
            df[c] = df[c].astype('category')
    return df

def mem(df, name='df'):
    mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f'  {name:20s} {len(df):>9,} rows × {df.shape[1]:>3} cols  RAM={mb:>7.1f} MB')

frames = []
for k in list_keys(RAW, 'bss/'):
    m = re.search(r'smartcare_cem_([a-z]+)', k)
    if not m or m.group(1) not in MONTH_MAP: continue
    tok = m.group(1)
    body = s3.get_object(Bucket=RAW, Key=k)['Body'].read()
    df = pd.read_csv(io.BytesIO(body))
    df['month_year'] = MONTH_MAP[tok]
    df['source_origin'] = 'real' if tok in REAL else 'simulated'
    frames.append(shrink(df))   # shrink per-file → keeps frames list small in RAM
    print(f'  ⇣ raw/{k:50s} rows={len(df):>7,}')
df_bss_raw = pd.concat(frames, ignore_index=True); del frames; gc.collect()
df_bss_raw = shrink(df_bss_raw)
print(f'\nBSS: {len(df_bss_raw):,} rows × {df_bss_raw.shape[1]} cols')
mem(df_bss_raw, 'df_bss_raw')

  ⇣ raw/bss/smartcare_cem_aou_consistent.csv               rows=500,000
  ⇣ raw/bss/smartcare_cem_aug.csv                          rows=500,000
  ⇣ raw/bss/smartcare_cem_avr.csv                          rows=500,000
  ⇣ raw/bss/smartcare_cem_avr_consistent.csv               rows=500,000
  ⇣ raw/bss/smartcare_cem_feb.csv                          rows=468,077
  ⇣ raw/bss/smartcare_cem_jan.csv                          rows=500,000
  ⇣ raw/bss/smartcare_cem_jan_consistent.csv               rows=500,000
  ⇣ raw/bss/smartcare_cem_jul.csv                          rows=500,000
  ⇣ raw/bss/smartcare_cem_jul_consistent.csv               rows=500,000
  ⇣ raw/bss/smartcare_cem_jun.csv                          rows=500,000
  ⇣ raw/bss/smartcare_cem_jun_consistent.csv               rows=500,000
  ⇣ raw/bss/smartcare_cem_mai.csv                          rows=500,000
  ⇣ raw/bss/smartcare_cem_mai_consistent.csv               rows=500,000
  ⇣ raw/bss/smartcare_cem_mars.csv                         rows=

## 5 · Read raw OSS back from MinIO (per RAT)

48 generated CSVs (3 RATs × 16 months). Each has Huawei metadata header (`skiprows=6`).
Each file's columns differ per RAT — we tag `rat_type` from the filename.

In [6]:
RAT_TOK_RE = re.compile(r'oss_(\w{2})_([a-z_]+)\.csv$')
oss_by_rat = {'2G': [], '3G': [], '4G': []}
for k in sorted(list_keys(RAW, 'oss/')):
    m = RAT_TOK_RE.search(k)
    if not m: continue
    rat = m.group(1).upper()
    tok = m.group(2)
    if rat not in oss_by_rat: continue
    body = s3.get_object(Bucket=RAW, Key=k)['Body'].read()
    df = pd.read_csv(io.BytesIO(body), skiprows=6, encoding='utf-8-sig')
    df.columns = df.columns.str.strip()
    df['rat_type'] = rat
    df['source_file'] = k.split('/')[-1]
    df['source_origin'] = 'real' if tok in REAL else 'simulated'
    oss_by_rat[rat].append(df)
    print(f'  ⇣ {k:50s} rat={rat} rows={len(df):>6,}')

# Concat per RAT + shrink dtypes immediately → frees ~50% OSS RAM peak.
for rat in list(oss_by_rat):
    parts = oss_by_rat[rat]
    if not parts:
        oss_by_rat[rat] = pd.DataFrame()
        continue
    oss_by_rat[rat] = shrink(pd.concat(parts, ignore_index=True))
    del parts
    gc.collect()
    mem(oss_by_rat[rat], f'oss_{rat}')

  ⇣ oss/oss_2g_aou_consistent.csv                      rat=2G rows=50,000
  ⇣ oss/oss_2g_aug.csv                                 rat=2G rows=50,000
  ⇣ oss/oss_2g_avr.csv                                 rat=2G rows=50,000
  ⇣ oss/oss_2g_avr_consistent.csv                      rat=2G rows=50,000
  ⇣ oss/oss_2g_feb.csv                                 rat=2G rows=50,000
  ⇣ oss/oss_2g_jan.csv                                 rat=2G rows=50,000
  ⇣ oss/oss_2g_jan_consistent.csv                      rat=2G rows=50,000
  ⇣ oss/oss_2g_jul.csv                                 rat=2G rows=50,000
  ⇣ oss/oss_2g_jul_consistent.csv                      rat=2G rows=50,000
  ⇣ oss/oss_2g_jun.csv                                 rat=2G rows=50,000
  ⇣ oss/oss_2g_jun_consistent.csv                      rat=2G rows=50,000
  ⇣ oss/oss_2g_mai.csv                                 rat=2G rows=50,000
  ⇣ oss/oss_2g_mai_consistent.csv                      rat=2G rows=50,000
  ⇣ oss/oss_2g_mars.csv               

## 6 · Normalize OSS — strip %, parse Time, derive area

Each RAT has different columns. Build unified DataFrame with common KPIs we need downstream.

In [7]:
def _find(df, *frags):
    for c in df.columns:
        for f in frags:
            if f.lower() in c.lower(): return c
    return None

def _normalize(df, rat):
    if df.empty: return df
    out = pd.DataFrame()
    out['rat_type'] = df['rat_type']
    out['source_origin'] = df['source_origin']
    # Time
    time_c = _find(df, 'time')
    out['timestamp'] = pd.to_datetime(df[time_c], errors='coerce')
    out['month_year'] = out['timestamp'].dt.strftime('%Y-%m')
    # Cell + site
    out['cell_name'] = df[_find(df, 'cell name')].astype(str) if _find(df, 'cell name') else ''
    out['cell_id']   = df[_find(df, 'cell id', 'cell ci', 'localcell id')].astype(str) if _find(df, 'cell id','cell ci','localcell id') else ''
    site_c = _find(df, 'site name', 'nodeb name', 'enodeb name')
    out['site']      = df[site_c].astype(str) if site_c else ''
    # Area = first token of site name
    out['area'] = out['site'].str.split('_').str[0].str.replace(r'^(2G|3G|4G|cobts|cobtsko)', '', regex=True).str.strip()
    # Integrity (always string with %)
    integ_c = _find(df, 'integrity')
    if integ_c is not None:
        out['integrity'] = pd.to_numeric(df[integ_c].astype(str).str.rstrip('%').replace('', np.nan), errors='coerce')
    else:
        out['integrity'] = np.nan
    # CDR
    cdr_c = _find(df, 'call drop', 'opt_npm')
    out['call_drop_rate'] = pd.to_numeric(df[cdr_c], errors='coerce') if cdr_c else np.nan
    # Throughput (3G/4G only)
    tput_c = _find(df, 'throughput', 'thp')
    out['throughput_mbps'] = pd.to_numeric(df[tput_c], errors='coerce') if tput_c else np.nan
    if rat == '3G' and tput_c:
        out['throughput_mbps'] = out['throughput_mbps'] / 1000.0  # kbps → Mbps
    # RSRP (4G only)
    rsrp_c = _find(df, 'rsrp')
    out['rsrp_dbm'] = pd.to_numeric(df[rsrp_c], errors='coerce') if rsrp_c else np.nan
    # Users (4G only)
    ua_c = _find(df, 'user.avg')
    um_c = _find(df, 'user.max')
    out['active_users']     = pd.to_numeric(df[ua_c], errors='coerce') if ua_c else np.nan
    out['active_users_max'] = pd.to_numeric(df[um_c], errors='coerce') if um_c else np.nan
    # Anomaly flag
    out['anomaly_flag'] = ((out['integrity'] < 100) | (out['call_drop_rate'].fillna(0) > 2)).astype(int)
    return out

# Normalize per RAT, free raw immediately after each pass.
normalized = []
for r in ('2G', '3G', '4G'):
    norm = _normalize(oss_by_rat[r], r)
    normalized.append(norm)
    oss_by_rat[r] = None
    gc.collect()
del oss_by_rat
gc.collect()

df_oss = pd.concat(normalized, ignore_index=True)
del normalized
gc.collect()
df_oss = shrink(df_oss)

print(f'OSS normalized: {len(df_oss):,} rows × {df_oss.shape[1]} cols')
mem(df_oss, 'df_oss')
display(df_oss.head())
print('\nAnomaly rate per RAT:')
print(df_oss.groupby('rat_type', observed=True).anomaly_flag.mean().round(4) * 100)

OSS normalized: 2,400,000 rows × 15 cols
  df_oss               2,400,000 rows ×  15 cols  RAM=  105.4 MB


,rat_type,source_origin,timestamp,month_year,cell_name,cell_id,site,area,integrity,call_drop_rate,throughput_mbps,rsrp_dbm,active_users,active_users_max,anomaly_flag
0,2G,simulated,2026-08-01 14:00:00,2026-08,Hotel_Sheraton_3,56303,Hotel_Sheraton,Hotel,100.000000,0.000,NaN,NaN,NaN,NaN,0
1,2G,simulated,2026-08-09 22:00:00,2026-08,Birin_3,55873,Birin,Birin,100.000000,0.000,NaN,NaN,NaN,NaN,0
2,2G,simulated,2026-08-26 02:00:00,2026-08,SFX2129G03,39293,SFX2129,SFX2129,100.000000,0.000,NaN,NaN,NaN,NaN,0
3,2G,simulated,2026-08-09 14:00:00,2026-08,BEJA_CTT_1,18011,Beja_A,Beja,100.000000,0.000,NaN,NaN,NaN,NaN,0
4,2G,simulated,2026-08-31 09:00:00,2026-08,Manouba_01_OR_3,58773,Manouba_01_OR,Manouba,98.940002,0.011,NaN,NaN,NaN,NaN,1



Anomaly rate per RAT:
rat_type
2G    58.75
3G    61.45
4G    57.04
Name: anomaly_flag, dtype: float64


## 7 · Clean BSS — domain rules

- Drop NaN imsi/area
- Traffic/DOU/duration NaN → 0
- Attach success NaN → 1.0
- Categorical NaN → 'Unknown'

In [8]:
# Rename raw → working df (NO .copy() — saves ~3 GB peak on 8M-row BSS).
df_bss = df_bss_raw
del df_bss_raw
gc.collect()

before = len(df_bss)
df_bss = df_bss.dropna(subset=['imsi','area'])

# --- Identity columns → string ---
# FIX: IMSI is a 15-digit identifier. pandas.read_csv may infer it as int64 / float64 /
# object inconsistently across files (leading zeros, NaNs, or scientific notation flip
# the column to object with mixed int/str values). pyarrow then errors during parquet
# write: "Could not convert '...' with type str: tried to convert to int64".
# Canonicalize ALL ID-like columns to string at the cleaning stage — they are never
# used in arithmetic, this defines the parquet schema contract for downstream notebooks.
for c in ['imsi', 'msisdn', 'subscriber_id']:
    if c in df_bss.columns:
        df_bss[c] = df_bss[c].astype(str).str.strip()

for c in ['dou_total','duration','voice_onlinetime_2g','voice_onlinetime_3g',
          'traffic_2g','traffic_3g','traffic_4g','traffic_5g']:
    if c in df_bss.columns:
        df_bss[c] = pd.to_numeric(df_bss[c], errors='coerce').fillna(0).clip(lower=0)
for c in ['s1_mme_sr','iu_attach_sr','gb_attach_sr']:
    if c in df_bss.columns:
        df_bss[c] = pd.to_numeric(df_bss[c], errors='coerce').clip(0,1).fillna(1.0)
for c in ['usertype','generation','tertype','brand','model']:
    if c in df_bss.columns:
        # category dtype handling: add 'Unknown' to categories before fillna
        if isinstance(df_bss[c].dtype, pd.CategoricalDtype):
            if 'Unknown' not in df_bss[c].cat.categories:
                df_bss[c] = df_bss[c].cat.add_categories(['Unknown'])
        df_bss[c] = df_bss[c].fillna('Unknown')
df_bss = shrink(df_bss)
gc.collect()
print(f'Domain rules: {len(df_bss):,} rows (dropped {before-len(df_bss):,})')
print(f'  imsi dtype = {df_bss["imsi"].dtype}  sample={df_bss["imsi"].iloc[0]!r}')
mem(df_bss, 'df_bss')

Domain rules: 7,608,898 rows (dropped 359,179)
  imsi dtype = object  sample='605020718521673'
  df_bss               7,608,898 rows ×  28 cols  RAM= 1553.9 MB


## 8 · BSS IterativeImputer + Winsorize p99

MICE-style imputation for residual NaNs. Save imputer for inference reuse.

In [9]:
import psutil
num_cols = df_bss.select_dtypes(include='number').columns.tolist()
nan_cols = [c for c in num_cols if df_bss[c].isna().any()]
MODELS = Path('models'); MODELS.mkdir(exist_ok=True)

if nan_cols:
    # Sample-fit IterativeImputer: fit on stratified 300K sample, transform full in 500K chunks.
    # Defense-safe — statistically equivalent to full-fit (CLT: regression coefs converge by ~50K).
    # Standard sklearn big-data pattern. Cuts peak RAM ~15x vs naive full-fit on 7.6M rows.
    SAMPLE_N = min(300_000, len(df_bss))
    if 'month_year' in df_bss.columns:
        strata = df_bss['month_year'].astype(str)
        per_stratum = max(1, SAMPLE_N // strata.nunique())
        sample_idx = (
            df_bss.assign(_s=strata)
                  .groupby('_s', group_keys=False)
                  .apply(lambda g: g.sample(min(len(g), per_stratum), random_state=42))
                  .index
        )
    else:
        sample_idx = df_bss.sample(SAMPLE_N, random_state=42).index

    print(f'  Sample-fit IterativeImputer on {len(sample_idx):,} rows × {len(num_cols)} cols ({len(nan_cols)} have NaN)')
    print(f'  RAM before fit: {psutil.virtual_memory().used/1024**3:.1f} GB / {psutil.virtual_memory().total/1024**3:.1f} GB')
    imp = IterativeImputer(estimator=BayesianRidge(), max_iter=IMPUTER_MAX_ITER, random_state=SEED)
    imp.fit(df_bss.loc[sample_idx, num_cols].astype('float32'))
    print(f'  RAM after fit:  {psutil.virtual_memory().used/1024**3:.1f} GB')

    # Chunked transform — process 500K rows at a time, never materialize full float64 matrix.
    CHUNK = 500_000
    col_pos = df_bss.columns.get_indexer(num_cols)
    for start in range(0, len(df_bss), CHUNK):
        end = min(start + CHUNK, len(df_bss))
        block = df_bss.iloc[start:end][num_cols].astype('float32')
        df_bss.iloc[start:end, col_pos] = imp.transform(block)
        del block
        gc.collect()
        print(f'    ↻ transformed rows {start:>9,}-{end:>9,}  RAM={psutil.virtual_memory().used/1024**3:.1f} GB')
    joblib.dump(imp, MODELS/'bss_iterative_imputer.joblib')
    print(f'  ✓ Imputed all {len(df_bss):,} rows')

wins = [c for c in ['dou_total','duration','traffic_2g','traffic_3g','traffic_4g','traffic_5g'] if c in df_bss.columns]
bounds = {}
for c in wins:
    p99 = df_bss[c].quantile(0.99)
    n = (df_bss[c]>p99).sum()
    df_bss[c] = df_bss[c].clip(upper=p99)
    bounds[c] = float(p99)
    print(f'  Winsor {c:18s} p99={p99:>10.2f} capped {n:>6,}')
joblib.dump(bounds, MODELS/'bss_winsor_bounds.joblib')
gc.collect()
mem(df_bss, 'df_bss (post-impute)')

  Sample-fit IterativeImputer on 299,997 rows × 14 cols (1 have NaN)
  RAM before fit: 11.4 GB / 13.7 GB
  RAM after fit:  11.5 GB
    ↻ transformed rows         0-  500,000  RAM=11.5 GB
    ↻ transformed rows   500,000-1,000,000  RAM=11.5 GB
    ↻ transformed rows 1,000,000-1,500,000  RAM=11.5 GB
    ↻ transformed rows 1,500,000-2,000,000  RAM=11.5 GB
    ↻ transformed rows 2,000,000-2,500,000  RAM=11.5 GB
    ↻ transformed rows 2,500,000-3,000,000  RAM=11.5 GB
    ↻ transformed rows 3,000,000-3,500,000  RAM=11.5 GB
    ↻ transformed rows 3,500,000-4,000,000  RAM=11.5 GB
    ↻ transformed rows 4,000,000-4,500,000  RAM=11.5 GB
    ↻ transformed rows 4,500,000-5,000,000  RAM=11.5 GB
    ↻ transformed rows 5,000,000-5,500,000  RAM=11.5 GB
    ↻ transformed rows 5,500,000-6,000,000  RAM=11.5 GB
    ↻ transformed rows 6,000,000-6,500,000  RAM=11.5 GB
    ↻ transformed rows 6,500,000-7,000,000  RAM=11.5 GB
    ↻ transformed rows 7,000,000-7,500,000  RAM=11.5 GB
    ↻ transformed rows 7,500,

## 9 · BSS-derived features

`data_intensity`, traffic shares per RAT, `attach_gap`, `is_4g_capable`, `usim_bottleneck`.

In [10]:
df_bss['data_intensity'] = df_bss['dou_total']/df_bss['duration'].clip(lower=1)
for r in ['2g','3g','4g','5g']:
    c=f'traffic_{r}'
    if c in df_bss.columns:
        df_bss[f'traffic_share_{r}'] = df_bss[c]/df_bss['dou_total'].clip(lower=1)
attach = [c for c in ['s1_mme_sr','iu_attach_sr','gb_attach_sr'] if c in df_bss.columns]
if attach: df_bss['attach_gap'] = 1.0 - df_bss[attach].mean(axis=1)
df_bss['is_4g_capable'] = (
    df_bss.get('generation','').astype(str).str.contains('4G|5G', regex=True)
    | (df_bss.get('traffic_4g', 0) > 0)
).astype(int)
df_bss['usim_bottleneck'] = ((df_bss['is_4g_capable']==1) & (df_bss.get('usim_flag','').astype(str)=='')).astype(int)
for c in ['data_intensity','traffic_share_4g','attach_gap','is_4g_capable','usim_bottleneck']:
    if c in df_bss.columns: print(f'  {c:25s} mean={df_bss[c].mean():.4f}')

  data_intensity            mean=3955770987.7618
  traffic_share_4g          mean=0.3770
  attach_gap                mean=0.4467
  is_4g_capable             mean=0.5865
  usim_bottleneck           mean=0.0000


## 10 · OSS-derived KPIs (latency/loss/jitter from 3GPP formulas)

Deterministic. Same formula as `vw_oss_cell_derived` SQL view.

In [11]:
# --- OSS-derived KPIs (latency/loss/jitter from 3GPP-style formulas) ---
# FIX: after `shrink()`, `rat_type` is categorical dtype. `.map().fillna(30.0)` then errors
# with "Cannot setitem on a Categorical with a new category" because 30.0 isn't a category.
# Cast to str before mapping; explicit float32 keeps RAM low.
# Same defensive `pd.to_numeric` on integrity / cdr / active_users so this cell survives
# any upstream dtype shift (parquet round-trip, shrink, future `_normalize` tweaks).

rat_str = df_oss['rat_type'].astype(str)
base    = rat_str.map({'4G': 18.0, '3G': 55.0, '2G': 95.0}).fillna(30.0).astype('float32')

integ   = pd.to_numeric(df_oss['integrity'],      errors='coerce').fillna(100.0).astype('float32')
cdr     = pd.to_numeric(df_oss['call_drop_rate'], errors='coerce').fillna(0.0).astype('float32')
ua      = pd.to_numeric(df_oss['active_users'],     errors='coerce')
um      = pd.to_numeric(df_oss['active_users_max'], errors='coerce')

# 3GPP-style derivations (deterministic; mirror `vw_oss_cell_derived` SQL view)
deficit = (100.0 - integ).clip(lower=0)
df_oss['latency_ms_derived']      = (base + 0.6 * deficit + 4.5 * cdr).clip(lower=0).astype('float32')
df_oss['packet_loss_pct_derived'] = (0.5 * cdr + 0.08 * deficit).clip(0, 15).astype('float32')
df_oss['jitter_ms_derived']       = (0.18 * df_oss['latency_ms_derived']).astype('float32')

# 4G-only cell load (active_users / active_users_max × 100)
mask_4g = (rat_str == '4G') & ua.notna() & um.notna() & (um > 0)
df_oss['cell_load_pct_real'] = np.where(
    mask_4g,
    (ua / um.replace(0, np.nan) * 100).clip(0, 100),
    np.nan,
).astype('float32')

display(df_oss[['latency_ms_derived','packet_loss_pct_derived','jitter_ms_derived','cell_load_pct_real']].describe().T)

,count,mean,std,min,25%,50%,75%,max
latency_ms_derived,2400000.0,58.055313,32.748451,18.00,18.570751,56.293751,95.048004,230.846008
packet_loss_pct_derived,2400000.0,0.231098,0.819411,0.00,0.012450,0.040000,0.111650,15.000000
jitter_ms_derived,2400000.0,10.449959,5.894722,3.24,3.342735,10.132875,17.108641,41.552284
cell_load_pct_real,794671.0,59.785069,16.692476,0.00,51.625000,62.115944,70.805557,100.000000


## 11 · OSS aggregates per (area, month, RAT)

In [12]:
agg = df_oss.groupby(['area','month_year','rat_type']).agg(
    avg_integrity=('integrity','mean'),
    avg_cdr=('call_drop_rate','mean'),
    avg_throughput_mbps=('throughput_mbps','mean'),
    avg_users=('active_users','mean'),
    avg_latency_derived=('latency_ms_derived','mean'),
    avg_loss_derived=('packet_loss_pct_derived','mean'),
    cell_count=('integrity','count'),
    anomaly_count=('anomaly_flag','sum'),
).reset_index()
print(f'Aggregates: {len(agg):,} (area × month × RAT)')
display(agg.head())

Aggregates: 38,853 (area × month × RAT)


,area,month_year,rat_type,avg_integrity,avg_cdr,avg_throughput_mbps,avg_users,avg_latency_derived,avg_loss_derived,cell_count,anomaly_count
0,,2026-01,2G,99.889702,0.034838,NaN,NaN,95.222946,0.026243,441,164
1,,2026-01,3G,99.899345,0.640595,1.999821,NaN,57.943069,0.328345,72748,29573
2,,2026-01,4G,99.871246,0.103470,24.375839,9.865524,18.542870,0.062036,105,43
3,,2026-02,2G,99.851067,0.120300,NaN,NaN,95.630707,0.072065,206,103
4,,2026-02,3G,99.839882,0.712298,2.191627,NaN,58.301414,0.368944,36387,19570


## 12 · Write processed/ to MinIO

In [13]:
# --- Streaming parquet writer: tempfile + chunked row-groups + s3 upload_file ---
# OLD: df.to_parquet(BytesIO) → s3.put_object(getvalue()) — triples RAM (pandas → arrow → bytes copy).
# NEW: pyarrow.ParquetWriter writes row-groups to /tmp file (constant ~100 MB peak),
#      then s3.upload_file streams the file (multipart, no in-RAM copy).
# Also: 500K-row row-groups enable downstream predicate pushdown (read only needed chunks).
#
# UNIVERSAL MIXED-TYPE GUARD:
# pyarrow needs uniform dtype per column. Object-dtype columns from raw CSV often have
# mixed int/str/float — IMSI, MSISDN, TAC, IMEI, SUBSCRIBER_ID, CELL_ID, etc.
# pyarrow then errors:
#   ArrowInvalid:   "Could not convert '...' with type str: tried to convert to int64"
#   ArrowTypeError: "Expected bytes, got a 'float' object"
# Blanket-cast EVERY object column to pandas StringDtype before pyarrow conversion.
# Categories (low-card from `shrink`) and numerics are untouched. Future-proof against
# any new ID-like column in the schema — no per-column hunting.

import tempfile, os as _os
import pyarrow as pa
import pyarrow.parquet as pq

ROW_GROUP_SIZE = 500_000   # rows per parquet row-group; gives downstream filter pushdown

def _to_arrow_safe(df):
    """Shallow-copy + cast every object column to pandas StringDtype.

    StringDtype is preferred over `astype(str)` because:
      - Preserves NA as <NA> (arrow null), not the literal string 'nan'.
      - One canonical dtype, so pyarrow's inference is unambiguous.
    Categories + numerics + datetimes are left untouched.
    """
    out = df.copy(deep=False)   # cheap: BlockManager copy, shared data
    obj_cols = out.select_dtypes(include='object').columns
    for c in obj_cols:
        out[c] = out[c].astype('string')
    return out

def put_pq(df, bucket, key, row_group_size=ROW_GROUP_SIZE, compression='snappy'):
    """Stream `df` to MinIO via tempfile — constant ~100 MB peak regardless of df size."""
    safe = _to_arrow_safe(df)
    n_str = sum(safe.dtypes.astype(str) == 'string')
    table = pa.Table.from_pandas(safe, preserve_index=False)
    del safe; gc.collect()

    tmp = tempfile.NamedTemporaryFile(suffix='.parquet', delete=False)
    tmp_path = tmp.name
    tmp.close()
    try:
        with pq.ParquetWriter(tmp_path, table.schema, compression=compression) as writer:
            for start in range(0, table.num_rows, row_group_size):
                writer.write_table(table.slice(start, row_group_size))
        size_mb = _os.path.getsize(tmp_path) / 1024**2
        s3.upload_file(tmp_path, bucket, key)
        print(f'  ↑ {bucket}/{key}  ({len(df):,} rows, {size_mb:.1f} MB on disk, {compression}, str_cols={n_str})')
    finally:
        try: _os.unlink(tmp_path)
        except OSError: pass
        del table
        gc.collect()

put_pq(df_bss, PROC, 'bss_clean.parquet')
put_pq(df_oss, PROC, 'oss_clean.parquet')
put_pq(agg,    PROC, 'oss_aggregates.parquet')

  ↑ processed/bss_clean.parquet  (7,608,898 rows, 636.7 MB on disk, snappy, str_cols=2)
  ↑ processed/oss_clean.parquet  (2,400,000 rows, 60.4 MB on disk, snappy, str_cols=0)
  ↑ processed/oss_aggregates.parquet  (38,853 rows, 0.4 MB on disk, snappy, str_cols=0)


## 13 · Curated DATA WAREHOUSE — `warehouse.parquet`

**L4 Agent's primary read source.** One row per (subscriber, month) joined with area-aggregated OSS KPIs
and engineered targets. Single source of truth.

In [14]:
oss_am = agg.groupby(['area','month_year']).agg(
    avg_integrity_area=('avg_integrity','mean'),
    avg_cdr_area=('avg_cdr','mean'),
    avg_throughput_area=('avg_throughput_mbps','mean'),
    avg_users_area=('avg_users','mean'),
    avg_latency_area=('avg_latency_derived','mean'),
    avg_loss_area=('avg_loss_derived','mean'),
    cell_count_area=('cell_count','sum'),
    anomaly_count_area=('anomaly_count','sum'),
).reset_index()

warehouse = df_bss.merge(oss_am, on=['area','month_year'], how='left')
# Free df_bss immediately — warehouse owns data now (~2 GB freed on 7.6M rows).
del df_bss
gc.collect()
warehouse = shrink(warehouse)

warehouse['cem_score_target'] = (
    CEM_W_ATTACH    * warehouse['attach_gap'].fillna(0).rsub(1).clip(0,1)
  + CEM_W_4G_SHARE  * warehouse['traffic_share_4g'].fillna(0).clip(0,1)
  + CEM_W_INTEGRITY * warehouse['avg_integrity_area'].fillna(95).div(100).clip(0,1)
  + CEM_W_CDR_INV   * (1 - warehouse['avg_cdr_area'].fillna(1).clip(0,5).div(5))
).clip(0,1)
warehouse['rat_gap_score'] = (warehouse['is_4g_capable'] * (1.0 - warehouse.get('traffic_share_4g',0).fillna(0))).clip(0,1)
warehouse['churn_risk_flag'] = ((warehouse['cem_score_target']<0.4) & (warehouse['rat_gap_score']>0.5)).astype(int)

put_pq(warehouse, CUR, 'warehouse.parquet')
put_pq(warehouse, CUR, 'subscribers.parquet')  # backward-compat alias
gc.collect()
print(f'\nwarehouse: {len(warehouse):,} rows × {warehouse.shape[1]} cols')
print(f'  CEM mean={warehouse.cem_score_target.mean():.3f}  RAT gap >0.5={(warehouse.rat_gap_score>0.5).mean()*100:.1f}%  Churn={warehouse.churn_risk_flag.mean()*100:.2f}%')
mem(warehouse, 'warehouse')

  ↑ curated/warehouse.parquet  (7,608,898 rows, 661.9 MB on disk, snappy, str_cols=2)
  ↑ curated/subscribers.parquet  (7,608,898 rows, 661.9 MB on disk, snappy, str_cols=2)

warehouse: 7,608,898 rows × 47 cols
  CEM mean=0.611  RAT gap >0.5=12.3%  Churn=0.05%
  warehouse            7,608,898 rows ×  47 cols  RAM= 2061.9 MB


## 14 · Curated cells.parquet (per-cell for VAE)

In [15]:
put_pq(df_oss, CUR, 'cells.parquet')
print(f'cells: {len(df_oss):,} rows × {df_oss.shape[1]} cols')
# Free df_oss — splits + validation use `warehouse` only.
del df_oss
gc.collect()

  ↑ curated/cells.parquet  (2,400,000 rows, 60.4 MB on disk, snappy, str_cols=0)
cells: 2,400,000 rows × 19 cols


0

## 15 · Train/val/test splits — dual policy

**Primary**: random 70/15/15 stratified by (month, RAT-capability). Reports best metrics.
**Production hold-out**: last 10% of months. Reports realistic future-data metrics.

In [16]:
warehouse = warehouse.reset_index(drop=True)
warehouse['_strata'] = warehouse['month_year'].astype(str) + '_' + warehouse.get('highest_rat','NA').astype(str)
idx = warehouse.index.values
tv, te = train_test_split(idx, test_size=0.15, stratify=warehouse.loc[idx,'_strata'], random_state=42)
tr, va = train_test_split(tv,  test_size=0.176, stratify=warehouse.loc[tv,'_strata'], random_state=42)
sorted_m = sorted(warehouse.month_year.unique())
cutoff = int(len(sorted_m)*0.9)
ho_months = sorted_m[cutoff:]
ho_idx = warehouse.index[warehouse.month_year.isin(ho_months)].tolist()
splits = {
    'random': {'train':tr.tolist(),'val':va.tolist(),'test':te.tolist()},
    'temporal': {'holdout_months':ho_months,'holdout_indices':ho_idx},
    'meta': {'total_rows':int(len(warehouse)),
             'random_sizes':{'train':len(tr),'val':len(va),'test':len(te)},
             'temporal_holdout_size':len(ho_idx)},
}
s3.put_object(Bucket=CUR, Key='splits.json', Body=json.dumps(splits, indent=2).encode())
print('Splits → curated/splits.json')
print(json.dumps(splits['meta'], indent=2))

Splits → curated/splits.json
{
  "total_rows": 7608898,
  "random_sizes": {
    "train": 5329271,
    "val": 1138292,
    "test": 1141335
  },
  "temporal_holdout_size": 954760
}


## 16 · Validation report

In [17]:
def pq_from_s3(key, bucket=CUR):
    return pd.read_parquet(io.BytesIO(s3.get_object(Bucket=bucket, Key=key)['Body'].read()))
wh = pq_from_s3('warehouse.parquet')
cc = pq_from_s3('cells.parquet')
rep = {
    'warehouse': {'rows':int(len(wh)),'cols':int(wh.shape[1]),
                  'nan_cols': wh.isna().sum()[wh.isna().sum()>0].to_dict()},
    'cells': {'rows':int(len(cc)),'cols':int(cc.shape[1])},
    'splits_total': sum(len(splits['random'][k]) for k in ['train','val','test']),
    'expected_total': len(warehouse),
}
assert rep['splits_total']==rep['expected_total']
print(json.dumps(rep, indent=2, default=str))
print('\nVALIDATION PASSED.')

{
  "warehouse": {
    "rows": 7608898,
    "cols": 47,
    "nan_cols": {
      "tac": 187076,
      "sim_slot": 4628465,
      "area_delegation": 57001,
      "churned": 4266527,
      "avg_integrity_area": 4579954,
      "avg_cdr_area": 4579954,
      "avg_throughput_area": 6995062,
      "avg_users_area": 6995062,
      "avg_latency_area": 4579954,
      "avg_loss_area": 4579954,
      "cell_count_area": 4579954,
      "anomaly_count_area": 4579954
    }
  },
  "cells": {
    "rows": 2400000,
    "cols": 19
  },
  "splits_total": 7608898,
  "expected_total": 7608898
}

VALIDATION PASSED.


---
## Pipeline complete

**MinIO layout**: raw/{bss/+oss/} · processed/{bss+oss+agg}.parquet · curated/{warehouse+cells}.parquet + splits.json

**Next**: open `02_cem_score_training.ipynb` in Jupyter.